# 4.1.4 — Self-Play Data Quality Validation

Runs a small batch of self-play games and checks that the generated data has the structural properties required for policy network training.

**Pass criteria:**
1. All maps and modes sampled uniformly — no single (map, mode) pair dominates.
2. P1/P2 canonical assignment is exactly 50/50 across games (deterministic alternation).
3. Terminal win probabilities are centred at **0.5 ± 0.05** — symmetric agents should produce balanced games.
4. Every record has `state.whose_turn == 'mine'` and its `visit_dist` sums to ≤ 1.0.
5. Each game's 6 picks are all distinct (no brawler picked twice).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from bsdraft.mcts.evaluator import FMEvaluator
from bsdraft.fm.model import FMInference
from bsdraft.data.matchup_db import MatchupDB
from bsdraft.selfplay.generate import run_self_play_batch, ReplayBuffer, load_map_mode_pairs
from bsdraft.data.prep import SEASON_CONFIGS

SEASON   = "s48"
DATA_DIR = Path("..").resolve() / "data" / SEASON
N_GAMES  = 50
N_SIMS   = 2_000   # default; reduce to 500 for a faster smoke-check

evaluator = FMEvaluator(FMInference.load(DATA_DIR / "fm_model.pkl"))
db        = MatchupDB.load(DATA_DIR / "matchup_db.pkl")
pairs     = load_map_mode_pairs(SEASON_CONFIGS[SEASON]["db_path"])

print(f"{SEASON}  |  {len(pairs)} valid (map, mode) pairs  |  {N_GAMES} games @ {N_SIMS} sims/pick")

games = run_self_play_batch(
    n_games=N_GAMES,
    data_dir=DATA_DIR,
    map_mode_pairs=pairs,
    evaluator=evaluator,
    db=db,
    n_workers=1,
    seed=42,
    n_sims_per_pick=N_SIMS,
)

all_records = [r for g in games for r in g]
print(f"Generated {len(games)} games  →  {len(all_records)} records total")


## Game distribution

Maps and modes should be sampled uniformly — no map should dominate. P1/P2 canonical assignment is determined by `game_id % 2`, so with 50 games we expect exactly 25 P1-first and 25 P2-first games. The terminal win probability (P1's FM win prob at each game's final state) should be centred near 0.5 — if one team is systematically winning it suggests a bug in the symmetric-agent logic.

In [ ]:
# ── Map / mode counts ──────────────────────────────────────────────────────────
map_counts  = Counter(r.state.map_name for r in all_records if r.pick_number == 0)
mode_counts = Counter(r.state.mode     for r in all_records if r.pick_number == 0)

print("Maps played (one entry per game):")
for m, n in sorted(map_counts.items(), key=lambda x: -x[1]):
    print(f"  {m:<28} {n:>3}")

print(f"\nModes played: {dict(sorted(mode_counts.items()))}")

# ── P1/P2 balance ──────────────────────────────────────────────────────────────
# game_id % 2 == 0 → P1-canonical; game_id % 2 == 1 → P2-canonical.
game_ids  = sorted({r.game_id for r in all_records})
p1_games  = sum(1 for gid in game_ids if gid % 2 == 0)
p2_games  = sum(1 for gid in game_ids if gid % 2 == 1)
print(f"\nP1/P2 canonical balance: {p1_games} P1-first  /  {p2_games} P2-first  (expect {N_GAMES//2}/{N_GAMES//2})")

# ── Terminal win probabilities ─────────────────────────────────────────────────
# For each game, use the pick_number=0 record's terminal_win_prob.
# Pick 0 is always P1's pick, so win_prob = P(P1 wins), regardless of canonical perspective.
p1_win_probs = [
    next(r.terminal_win_prob for r in g if r.pick_number == 0)
    for g in games
]
mean_wp = float(np.mean(p1_win_probs))
std_wp  = float(np.std(p1_win_probs))
print(f"\nP1 terminal win probs:  mean={mean_wp:.3f}  std={std_wp:.3f}")

# ── Figure: map distribution + win-prob histogram ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

maps_sorted = sorted(map_counts, key=lambda m: map_counts[m])
axes[0].barh(maps_sorted, [map_counts[m] for m in maps_sorted], color="steelblue")
axes[0].axvline(N_GAMES / len(pairs), color="tomato", linestyle="--", label="uniform expected")
axes[0].set_xlabel("games")
axes[0].set_title("Map sampling distribution")
axes[0].legend(fontsize=8)

axes[1].hist(p1_win_probs, bins=15, color="steelblue", edgecolor="white")
axes[1].axvline(0.5, color="tomato", linestyle="--", label="0.5 (symmetric)")
axes[1].axvline(mean_wp, color="orange", linestyle="-", label=f"mean={mean_wp:.3f}")
axes[1].set_xlabel("P1 terminal win probability")
axes[1].set_title("Terminal win prob distribution")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


## Record validity checks

Every `SelfPlayRecord` must satisfy structural invariants before it can be used for training:
- `state.whose_turn == 'mine'`: picks are always stored from the acting agent's perspective.
- `visit_dist` sums to ≤ 1.0 (it is the top-N subset, so slightly below 1 is normal).
- `pick_made` is not already on either team at the time of the pick (no duplicate picks in a game).

Failures here indicate a bug in the game runner's state mirroring or pick application logic.

In [ ]:
vocab = set(evaluator._fm.schema.vocab)
errors = []

for g_idx, game in enumerate(games):
    picks_so_far = set()
    for r in sorted(game, key=lambda x: x.pick_number):

        if r.state.whose_turn != "mine":
            errors.append(f"game {r.game_id} pick {r.pick_number}: whose_turn={r.state.whose_turn!r} (expected 'mine')")

        dist_sum = sum(r.visit_dist.values())
        if dist_sum > 1.001 or dist_sum <= 0.0:
            errors.append(f"game {r.game_id} pick {r.pick_number}: visit_dist sums to {dist_sum:.4f}")

        if r.pick_made not in vocab:
            errors.append(f"game {r.game_id} pick {r.pick_number}: pick_made {r.pick_made!r} not in vocab")

        if r.pick_made in picks_so_far:
            errors.append(f"game {r.game_id} pick {r.pick_number}: duplicate pick {r.pick_made!r}")
        picks_so_far.add(r.pick_made)

    if len(picks_so_far) != 6:
        errors.append(f"game {g_idx}: expected 6 unique picks, got {len(picks_so_far)}")

if errors:
    print(f"FAIL — {len(errors)} error(s) found:")
    for e in errors[:10]:
        print(f"  {e}")
else:
    print(f"✓  All {len(all_records)} records pass validity checks")

# ── Pass criteria assertions ───────────────────────────────────────────────────
assert not errors, f"{len(errors)} record validity error(s) — see above"

assert p1_games == p2_games == N_GAMES // 2, (
    f"FAIL — P1/P2 balance: {p1_games} P1 / {p2_games} P2 (expected {N_GAMES//2} each)"
)
print(f"✓  P1/P2 balance exact: {p1_games}/{p2_games}")

assert abs(mean_wp - 0.5) <= 0.05, (
    f"FAIL — mean P1 win prob {mean_wp:.3f} outside 0.45–0.55. "
    "Symmetric agents should produce balanced games."
)
print(f"✓  Mean P1 win prob {mean_wp:.3f} within 0.5 ± 0.05")


## Example game

Print the full 6-pick sequence of one game: the board state before each pick, the brawler chosen, and the top-3 entries in the MCTS visit distribution. The visit distribution should be non-uniform — if all fractions are roughly equal the MCTS is not distinguishing between picks (too few sims or a flat FM).

In [ ]:
EXAMPLE_GAME_IDX = 0   # change to inspect different games

game = sorted(games[EXAMPLE_GAME_IDX], key=lambda r: r.pick_number)
r0   = game[0]
print(f"Game {r0.game_id}  |  {r0.state.map_name} ({r0.state.mode})"
      f"  |  skill_ns={r0.state.skill_ns:.1f}"
      f"  |  canonical={'P1' if r0.game_id % 2 == 0 else 'P2'}-first")
print(f"  terminal win prob (P1): {game[0].terminal_win_prob if game[0].state.is_first_pick else 1 - game[0].terminal_win_prob:.3f}")
print()
print(f"  {'#':<3} {'Team':<5} {'Pick':<16} {'Win prob':<10} Top-3 visit distribution")
print("  " + "─" * 72)

for r in game:
    team = "P1" if r.state.is_first_pick else "P2"
    top3 = sorted(r.visit_dist.items(), key=lambda x: -x[1])[:3]
    top3_str = "  ".join(f"{b}={v:.3f}" for b, v in top3)
    my_picks  = sorted(r.state.my_team)  if r.state.my_team  else ["—"]
    opp_picks = sorted(r.state.opp_team) if r.state.opp_team else ["—"]
    print(f"  {r.pick_number:<3} {team:<5} {r.pick_made:<16} {r.terminal_win_prob:<10.3f} {top3_str}")
    print(f"       my_team={my_picks}  opp_team={opp_picks}")

# Sanity: top-1 in visit_dist at each pick should be a reasonable choice
# (not necessarily pick_made, since we sample with temperature=1)
visit_dist_sums = [sum(r.visit_dist.values()) for r in game]
print(f"\n  visit_dist sums: {[f'{s:.3f}' for s in visit_dist_sums]}")
print(f"  (top-N subset; values < 1.0 are expected — remaining mass on lower-ranked brawlers)")


---

# 4.2.5 — Policy Network Training & Prior Quality

Trains a `PolicyNet` on the self-play records generated above, plots the training convergence, and validates that the learned policy produces meaningful pick distributions.

**Pass criteria:**
1. **Convergence** — validation KL loss decreases across epochs (model is learning).
2. **Quality** — policy top-5 at mid-draft states contains plausible picks for the map/mode, not a near-uniform spread.
3. **Integration** — `recommend()` runs without errors when `policy=` is passed; visit distributions with the policy prior differ measurably from the counter-rate prior.

> **Note on data scale:** 50 games × 6 records = 300 training examples is the > *minimum viable* amount. Expect partial fit, not full convergence. The cells below > validate the training *mechanics* and MCTS integration. Full convergence requires > 1,000+ games (Part 4.3 iterative self-play).

## Training curve

**What to look for:**
- Val loss should decrease from epoch 0 — if it is flat the model is not learning (check that `all_records` is non-empty and visit distributions are non-trivial).
- A gap between train and val is expected at small data scale; it will narrow with more games.
- Early stopping (dotted line) means the best checkpoint was restored before overfitting set in.

In [ ]:
from bsdraft.selfplay.policy_net import train_policy, POLICY_EXTRA_FEATURES

schema       = evaluator._fm.schema
loss_history = []

print(f"Training on {len(all_records)} records  |  {len(schema.vocab)} brawlers  "
      f"|  input dim = {schema.n_features + POLICY_EXTRA_FEATURES}")
policy = train_policy(
    all_records, schema,
    max_epochs=30, patience=5,
    verbose=False, loss_history=loss_history,
)
n_params = sum(p.numel() for p in policy.net.parameters())
best_val  = min(v for _, _, v in loss_history)
print(f"Trained {len(loss_history)} epochs  |  best val KL = {best_val:.4f}  "
      f"|  {n_params:,} parameters")

# ── Loss curves ───────────────────────────────────────────────────────────────
epochs     = [e for e, _, _ in loss_history]
train_loss = [t for _, t, _ in loss_history]
val_loss   = [v for _, _, v in loss_history]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(epochs, train_loss, lw=1.8, color="steelblue", label="Train")
ax.plot(epochs, val_loss,   lw=1.8, color="tomato",    label="Val",  linestyle="--")
ax.axvline(epochs[val_loss.index(best_val)], color="gray", lw=0.8,
           linestyle=":", label=f"best val epoch {epochs[val_loss.index(best_val)]}")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy (KL proxy) — lower is better")
ax.set_title("Policy network training convergence")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## Policy predictions vs MCTS visit distributions

For one record at each pick depth (1, 3, 5), print:
- **Policy top-5**: the network's predicted pick probabilities after training.
- **MCTS visit top-5**: the MCTS visit fractions that formed the training label.

They should be *roughly* aligned — exact match is not expected at 50 games. What matters is that the policy top-5 is non-uniform and contains recognisable brawlers for the map/mode, not a random spread. Matches (←) between the two columns confirm the policy is distilling MCTS knowledge.

In [ ]:
from bsdraft.mcts.state import available_actions

vocab = tuple(evaluator._fm.schema.vocab)

# One record at early, mid, and late pick depth
display_records = {}
for r in all_records:
    d = r.pick_number
    if d in (1, 3, 5) and d not in display_records:
        display_records[d] = r
    if len(display_records) == 3:
        break

for depth in sorted(display_records):
    rec   = display_records[depth]
    avail = available_actions(rec.state, vocab)

    pol_probs = policy.predict_prior(rec.state, avail)
    top_pol   = sorted(zip(avail, pol_probs), key=lambda x: -x[1])[:5]
    top_mcts  = sorted(rec.visit_dist.items(), key=lambda x: -x[1])[:5]

    my_str  = sorted(rec.state.my_team)  or ["—"]
    opp_str = sorted(rec.state.opp_team) or ["—"]
    print(f"Pick {depth}  |  {rec.state.map_name} ({rec.state.mode})"
          f"  my={my_str}  opp={opp_str}")
    print(f"  {'Policy top-5':<34}  MCTS visit top-5")
    print(f"  {'─'*33}   {'─'*28}")
    for i in range(5):
        pb, pp = top_pol[i]  if i < len(top_pol)  else ("—", 0.0)
        mb, mp = top_mcts[i] if i < len(top_mcts) else ("—", 0.0)
        match  = " ←" if pb == mb else ""
        print(f"  {pb:<18} {pp:.4f}    {mb:<18} {mp:.4f}{match}")
    print()


## MCTS with policy prior vs counter-rate prior

Run `recommend()` from the same mid-draft state under two PUCT prior configs:
- **Policy prior**: learned visit-distribution prior from the trained `PolicyNet`.
- **Counter-rate prior**: heuristic blend of pick-rate + avg counter-rate (Part 3.2).

**Pass criteria:**
- Both calls complete without error — confirms the integration is working end-to-end.
- Top-1 visit fractions and rankings may differ; differences should be interpretable (not noise).
- With only 50 training games the policy is underfit — stark differences are not expected. The comparison becomes more meaningful after a full self-play iteration.

In [ ]:
from bsdraft.mcts.recommend import recommend
import numpy as np

# Use the pick-3 held-out record as comparison state.
# Falls back to pick-2 or any available depth if pick-3 is not available.
rec_cmp = display_records.get(3) or display_records.get(2) or list(display_records.values())[0]
s = rec_cmp.state
whose = "my" if s.whose_turn == "mine" else "opponent's"
print(f"State: {s.map_name} ({s.mode})  |  pick {s.pick_number}  "
      f"|  {'P1' if s.is_first_pick else 'P2'}  |  → {whose} pick")
print(f"  my_team={sorted(s.my_team) or ['—']}   opp_team={sorted(s.opp_team) or ['—']}\n")

N_SIMS = 3_000
base_kw = dict(
    my_picks=list(s.my_team), opp_picks=list(s.opp_team),
    mode=s.mode, map_name=s.map_name, skill_ns=s.skill_ns,
    is_first_pick=s.is_first_pick, n_simulations=N_SIMS,
    evaluator=evaluator, db=db,
)

res_pol = recommend(**base_kw, rng=np.random.default_rng(7), policy=policy)
res_ctr = recommend(**base_kw, rng=np.random.default_rng(7), policy=None)

# ── Side-by-side top-5 ─────────────────────────────────────────────────────────
print(f"{'Rank':<5} {'Policy prior':<35}  {'Counter-rate prior'}")
print("─" * 76)
for i, (p, c) in enumerate(zip(res_pol.top_picks[:5], res_ctr.top_picks[:5])):
    same = "=" if p["brawler"] == c["brawler"] else " "
    print(f"  {i+1}   {p['brawler']:<18} {p['visit_fraction']*100:5.1f}%"
          f"  {same}  {c['brawler']:<18} {c['visit_fraction']*100:5.1f}%")

# ── Concentration (top-1 visit share) ─────────────────────────────────────────
top1_p = res_pol.top_picks[0]["visit_fraction"]
top1_c = res_ctr.top_picks[0]["visit_fraction"]
print(f"\nTop-1 visit fraction:  policy={top1_p:.3f}  counter-rate={top1_c:.3f}  "
      f"Δ={top1_p - top1_c:+.3f}")
print(f"{'→ Policy more concentrated' if top1_p > top1_c else '→ Counter-rate more concentrated'}")
print(f"\nPolicy confidence  : {res_pol.layer2['confidence_label']}")
print(f"Counter confidence : {res_ctr.layer2['confidence_label']}")


---

# Tree Vocab Pruning — Choosing `min_pick_rate`

Setting `min_pick_rate` removes rarely-picked brawlers from the **MCTS search tree**, reducing the branching factor without meaningfully affecting recommendations. Brawlers below the threshold are genuinely irrelevant on that specific map/mode/tier — they appear in fewer than 1-in-N games there.

**How to read the analysis below:**
- The *pick-rate distribution* shows how concentrated usage is across brawlers on each map.
- The *percentile table* maps thresholds → average number of brawlers kept per map.
- The *per-map breakdown* shows exactly which brawlers would be excluded so you can sanity-check.

**Season 48 numbers (100 brawlers, 26 maps, mid-tier):**

| `min_pick_rate` | Avg brawlers kept | % of vocab | Notes |
|---|---|---|---|
| 0.000 | 100 | 100% | No filtering |
| 0.003 | ~50 | ~50% | Conservative |
| **0.005** | **~44** | **~44%** | **← default / recommended** |
| 0.010 | ~30 | ~30% | Aggressive — may miss fringe viable picks |
| 0.020 | ~16 | ~16% | Very tight; only S/A-tier brawlers per map |

**Recommendation:** Start with `min_pick_rate = 0.005` (the code default).  It cuts the tree roughly in half to ~44 brawlers while keeping all brawlers with a meaningful presence on that map.  Use the bar chart below to verify no important brawler is excluded before tightening further.

In [ ]:
from bsdraft.data.matchup_db import skill_ns_to_tier
import pandas as pd

TIER = skill_ns_to_tier(2.0, db.skill_tier_boundaries)  # mid-tier for analysis

# Collect pick rates for every (brawler, map, mode) at mid-tier.
rows = []
for map_name, mode in pairs:
    for brawler in evaluator._fm.schema.vocab:
        entry = db.brawler_lookup(brawler, mode, map_name, TIER)
        rate  = entry["pick_rate"] if entry is not None else 0.0
        rows.append({"brawler": brawler, "map": map_name, "mode": mode, "pick_rate": rate})

df = pd.DataFrame(rows)

# ── Overall pick-rate distribution ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: histogram of all (brawler, map) pick rates
axes[0].hist(df["pick_rate"], bins=60, color="steelblue", edgecolor="none", log=True)
for thresh, col in [(0.005, "tomato"), (0.01, "orange"), (0.02, "gold")]:
    axes[0].axvline(thresh, color=col, lw=1.5, linestyle="--", label=f"{thresh:.3f}")
axes[0].set_xlabel("pick_rate (fraction of games on that map/mode)")
axes[0].set_ylabel("count (log scale)")
axes[0].set_title("Pick-rate distribution  (all brawlers × all maps, mid-tier)")
axes[0].legend(title="min_pick_rate", fontsize=8)

# Right: CDF — fraction of (brawler, map) pairs included at each threshold
sorted_rates = df["pick_rate"].sort_values().values
cdf_x = sorted_rates
cdf_y = np.arange(1, len(sorted_rates) + 1) / len(sorted_rates)
axes[1].plot(cdf_x, 1 - cdf_y, color="steelblue", lw=2)
for thresh, col in [(0.005, "tomato"), (0.01, "orange"), (0.02, "gold")]:
    frac_included = (df["pick_rate"] >= thresh).mean()
    axes[1].axvline(thresh, color=col, lw=1.5, linestyle="--",
                    label=f"{thresh:.3f} → {frac_included*100:.0f}% brawler-map pairs kept")
axes[1].set_xlabel("min_pick_rate threshold")
axes[1].set_ylabel("fraction of brawler-map pairs included")
axes[1].set_title("CDF: fraction kept at each threshold")
axes[1].legend(fontsize=8)
axes[1].set_xlim(0, 0.06)

plt.tight_layout()
plt.show()

# ── Percentile table ───────────────────────────────────────────────────────────
# How many unique brawlers are above threshold on the AVERAGE map?
thresholds = [0.0, 0.001, 0.003, 0.005, 0.008, 0.01, 0.015, 0.02, 0.03, 0.05]
vocab_size = len(evaluator._fm.schema.vocab)

print(f"\nTree vocab size at each min_pick_rate  (avg across {len(pairs)} maps, mid-tier)")
print(f"  {'threshold':>12}  {'avg brawlers kept':>18}  {'% of vocab':>10}  {'excluded example brawlers'}")
print("  " + "─"*80)
for t in thresholds:
    avg_kept = df.groupby(["map", "mode"]).apply(
        lambda g: (g["pick_rate"] >= t).sum()
    ).mean()
    excluded = (
        df[df["pick_rate"] < t]
        .groupby("brawler")["pick_rate"].mean()
        .sort_values(ascending=False)  # highest among excluded first
        .head(4).index.tolist()
    )
    excl_str = ", ".join(excluded) if excluded else "none"
    marker = " ◄ recommended" if t == 0.005 else ""
    print(f"  {t:>12.3f}  {avg_kept:>18.1f}  {avg_kept/vocab_size*100:>9.0f}%  {excl_str}{marker}")


In [ ]:
# ── Per-map breakdown for the chosen threshold ─────────────────────────────────
# Shows exactly which brawlers are excluded on a specific map so you can
# sanity-check that the threshold isn't cutting anything important.

INSPECT_MAP, INSPECT_MODE = pairs[0]   # change to any map you care about
THRESHOLD = 0.003                       # ◄ adjust to compare options

map_df = df[(df["map"] == INSPECT_MAP) & (df["mode"] == INSPECT_MODE)].copy()
map_df = map_df.sort_values("pick_rate", ascending=False).reset_index(drop=True)
map_df["kept"] = map_df["pick_rate"] >= THRESHOLD

kept     = map_df[map_df["kept"]]
excluded = map_df[~map_df["kept"]]

print(f"Map: {INSPECT_MAP} ({INSPECT_MODE})  |  threshold={THRESHOLD}")
print(f"  Kept:     {len(kept)} brawlers   (top {len(kept)/len(map_df)*100:.0f}%)")
print(f"  Excluded: {len(excluded)} brawlers  (bottom {len(excluded)/len(map_df)*100:.0f}%)")
print()

fig, ax = plt.subplots(figsize=(14, 5))
colors = ["steelblue" if k else "lightcoral" for k in map_df["kept"]]
ax.bar(range(len(map_df)), map_df["pick_rate"], color=colors, width=0.8)
ax.axhline(THRESHOLD, color="black", lw=1.2, linestyle="--", label=f"threshold={THRESHOLD}")
ax.set_xticks(range(len(map_df)))
ax.set_xticklabels(map_df["brawler"], rotation=90, fontsize=7)
ax.set_ylabel("pick_rate")
ax.set_title(f"Pick rates on {INSPECT_MAP} ({INSPECT_MODE}) — blue=kept, red=excluded")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nExcluded brawlers (pick_rate < {THRESHOLD}):")
for _, row in excluded.iterrows():
    print(f"  {row['brawler']:<18}  pick_rate={row['pick_rate']:.4f}")


---

# Ban Simulation Validation

Self-play games now sample probabilistic bans at game start.  The top-N brawlers by pick_rate on the map/mode/tier are each banned with probability `ban_p` (default 0.35), capped at `max_bans` (default 6).  

**Pass criteria:**
- Dominant brawlers (high pick_rate) are banned most frequently.
- Fringe brawlers are never banned.
- Bans propagate correctly into all 6 pick slots of a game.

In [ ]:
from bsdraft.selfplay.generate import sample_bans

# ── Ban frequency across 500 simulated games on one map/mode ───────────────────
from collections import Counter

SAMPLE_MAP, SAMPLE_MODE = pairs[0]
SAMPLE_TIER = skill_ns_to_tier(2.0, db.skill_tier_boundaries)

ban_counts = Counter()
n_trials = 500
for seed in range(n_trials):
    bans = sample_bans(
        evaluator._fm.schema.vocab, db,
        SAMPLE_MODE, SAMPLE_MAP, SAMPLE_TIER,
        np.random.default_rng(seed),
    )
    ban_counts.update(bans)

# Join with pick_rate for this map/mode/tier to verify correlation
rates = {}
for b in evaluator._fm.schema.vocab:
    entry = db.brawler_lookup(b, SAMPLE_MODE, SAMPLE_MAP, SAMPLE_TIER)
    rates[b] = entry["pick_rate"] if entry is not None else 0.0

ban_freq = {b: ban_counts[b] / n_trials for b in evaluator._fm.schema.vocab}

# Sort by pick_rate to show the correlation
brawlers_sorted = sorted(rates, key=lambda b: -rates[b])
top_n = 30

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: pick_rate vs ban frequency for top-30 brawlers
top_brawlers = brawlers_sorted[:top_n]
x = range(top_n)
axes[0].bar(x, [ban_freq[b] for b in top_brawlers], color="tomato", alpha=0.8, label="ban freq")
ax2 = axes[0].twinx()
ax2.plot(x, [rates[b] for b in top_brawlers], "o-", color="steelblue", ms=4, lw=1.5, label="pick_rate")
axes[0].set_xticks(x)
axes[0].set_xticklabels(top_brawlers, rotation=90, fontsize=7)
axes[0].set_ylabel("ban frequency (500 trials)", color="tomato")
ax2.set_ylabel("pick_rate", color="steelblue")
axes[0].set_title(f"Ban frequency vs pick_rate — top-30 brawlers\n{SAMPLE_MAP} ({SAMPLE_MODE})")
axes[0].axhline(0.35, color="tomato", lw=0.8, linestyle="--", alpha=0.5, label="target ban_p=0.35")

# Right: scatter — pick_rate vs ban frequency for all brawlers
all_rates = [rates[b] for b in evaluator._fm.schema.vocab]
all_bfreq = [ban_freq[b] for b in evaluator._fm.schema.vocab]
axes[1].scatter(all_rates, all_bfreq, alpha=0.6, s=30, color="steelblue")
axes[1].set_xlabel("pick_rate (meta strength proxy)")
axes[1].set_ylabel("ban frequency")
axes[1].set_title("Correlation: pick_rate → ban frequency\n(should be positive — dominant brawlers banned more)")
plt.tight_layout()
plt.show()

never_banned = [b for b in evaluator._fm.schema.vocab if ban_freq[b] == 0]
top5_banned  = ban_counts.most_common(5)
print(f"Top-5 most-banned: {[(b, f'{n}/{n_trials}') for b, n in top5_banned]}")
print(f"Brawlers never banned: {len(never_banned)}/{len(evaluator._fm.schema.vocab)} "
      f"(should be ~{len(evaluator._fm.schema.vocab)-15} — all outside top-15 by pick_rate)")

# Pass: the top-5 banned should all be top-15 pick-rate brawlers
top15 = set(brawlers_sorted[:15])
top5_names = {b for b, _ in top5_banned}
assert top5_names <= top15, f"Unexpected brawlers in top-5 banned: {top5_names - top15}"
print("✓  Top-banned brawlers are all in the top-15 by pick_rate")


---

# 4.3 — Iterative Self-Play Training

**Yes, you should train now.**  The full pipeline is implemented.  Run the cell below to kick off iterative self-play training with the recommended defaults.

**Time estimates** (serial, 1 worker):
| Setting | Games/iter | Sims/pick | Time/iter |
|---|---|---|---|
| Quick test | 20 | 500 | ~2 min |
| Standard | 500 | 2 000 | ~15 min |
| Standard | 500 | 5 000 | ~35 min |
| High-quality | 1 000 | 5 000 | ~70 min |

With `n_workers=4` (multiprocessing), divide time by ~3.5.

**How it works:**
1. **Generate** N self-play games with MCTS + current policy (iter 0 = no policy).
2. **Train** a fresh policy network on all accumulated data.
3. **Evaluate** new policy vs. old: 200 head-to-head games on held-out maps.
4. **Promote** if new policy mean FM win prob ≥ 0.525.  Repeat.

The trained policy is saved to `data/{SEASON}/policy/policy_best.pkl` and can be loaded directly into `recommend()` via the `policy=` argument.

In [ ]:
from bsdraft.selfplay.train import IterationConfig, run_iteration_loop, load_training_log

# ── Configuration ──────────────────────────────────────────────────────────────
# Adjust these to your time budget.  For a quick end-to-end test use the
# "QUICK TEST" values; for production-quality training use "STANDARD".

# ── Training mode ─────────────────────────────────────────────────────────────
# "standard"   : real training — ~15-20 min/iter serial, ~5-7 min with 4 workers.
#                Run this to produce a genuinely useful policy.
# "quick_test" : smoke-test only — validates the full pipeline in ~2-3 min.
#                Use this once to confirm everything works, not for real training.

TRAINING_MODE = "standard"   # ← change to "quick_test" only to re-validate plumbing

if TRAINING_MODE == "standard":
    # Full training run.  Increase N_ITERATIONS beyond 3 freely — gains typically
    # plateau after 5-10 iterations, but you can keep going.  Each iteration
    # accumulates more data and retrains from scratch, so later iterations
    # benefit from the growing replay buffer as well as the better policy prior.
    N_ITERATIONS     = 5          # ← run more (8-10) for a stronger policy
    N_GAMES_PER_ITER = 500        # ← increase to 1000 for higher-quality training data
    N_SIMS_ITER0     = 2_000      # iter 0: no policy yet, 2k is sufficient
    N_SIMS_LATER     = 5_000      # iter 1+: policy prior makes extra sims valuable
    N_EVAL_GAMES     = 200        # head-to-head games for promotion decision
    N_SIMS_EVAL      = 10_000      # sims per pick during evaluation
    N_WORKERS        = 1          # ← set to 4+ if you have CPU cores to spare (~3.5× faster)

elif TRAINING_MODE == "quick_test":
    # Smoke-test only.  Confirms the pipeline runs end-to-end but produces
    # a meaningless policy.  Run this once, then switch to "standard".
    N_ITERATIONS     = 2
    N_GAMES_PER_ITER = 30
    N_SIMS_ITER0     = 500
    N_SIMS_LATER     = 500
    N_EVAL_GAMES     = 20
    N_SIMS_EVAL      = 200
    N_WORKERS        = 4

config = IterationConfig(
    n_games_per_iter=N_GAMES_PER_ITER,
    n_sims_iter0=N_SIMS_ITER0,
    n_sims_later=N_SIMS_LATER,
    n_eval_games=N_EVAL_GAMES,
    n_sims_eval=N_SIMS_EVAL,
    n_workers=N_WORKERS,
    min_pick_rate=0.005,        # ← recommended threshold from analysis above
    ban_top_n=15,               # top-15 brawlers eligible for ban sampling
    ban_p=0.35,                 # each top-15 brawler banned in ~35% of games
    max_bans=6,
    seed=42,
)

print(f"Training config: {N_ITERATIONS} iter × {N_GAMES_PER_ITER} games @ "
      f"{N_SIMS_ITER0}/{N_SIMS_LATER} sims  |  eval: {N_EVAL_GAMES} games @ {N_SIMS_EVAL} sims")
print(f"min_pick_rate={config.min_pick_rate}  ban_p={config.ban_p}  max_bans={config.max_bans}")


In [ ]:
# ── Run training ───────────────────────────────────────────────────────────────
# resume=True continues from a previous run if a training log already exists.
# Set resume=False to start fresh.

import time
t0 = time.perf_counter()

results = run_iteration_loop(
    n_iterations=N_ITERATIONS,
    data_dir=DATA_DIR,
    map_mode_pairs=pairs,
    config=config,
    evaluator=evaluator,
    db=db,
    resume=False,
)

print(f"\nTraining complete in {(time.perf_counter()-t0)/60:.1f} min")
print(f"Ran {len(results)} new iteration(s)")


---

# 4.3.3 — Iteration Improvement Results

**Pass criterion:** Iteration 1 policy scores above 0.5 mean FM win probability against the iteration 0 baseline.  The top-5 recommendations from the trained policy should be qualitatively sensible and shift measurably across iterations.

In [ ]:
# ── Load training log ──────────────────────────────────────────────────────────
log = load_training_log(DATA_DIR)

if not log:
    print("No training log found — run the training cells above first.")
else:
    print(f"Training log: {len(log)} iteration(s)")
    print()
    print(f"  {'Iter':>4}  {'Games':>6}  {'Total':>6}  {'Sims':>5}  "
          f"{'Val KL':>7}  {'Eval win':>9}  {'Promoted':>8}")
    print("  " + "─"*62)
    for r in log:
        promoted_str = "YES ✓" if r["promoted"] else "no"
        print(f"  {r['iteration']:>4}  {r['n_games_generated']:>6}  "
              f"{r['n_games_total']:>6}  {r['n_sims_per_pick']:>5}  "
              f"{r['val_kl']:>7.4f}  {r['eval_win_prob']:>9.4f}  {promoted_str:>8}")

    if len(log) >= 2:
        # Pass criterion check
        iter1_win = log[1]["eval_win_prob"]
        if iter1_win > 0.5:
            print(f"\n✓  Iteration 1 eval win prob = {iter1_win:.4f} > 0.5  (pass)")
        else:
            print(f"\n✗  Iteration 1 eval win prob = {iter1_win:.4f} ≤ 0.5  "
                  f"(may need more games or sims per pick)")


In [ ]:
# ── KL loss curves per iteration ───────────────────────────────────────────────
if log and any(r.get("loss_history") for r in log):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    colors = plt.cm.Blues(np.linspace(0.4, 0.95, len(log)))

    for idx, r in enumerate(log):
        hist = r.get("loss_history", [])
        if not hist:
            continue
        epochs     = [e[0] for e in hist]
        train_loss = [e[1] for e in hist]
        val_loss   = [e[2] for e in hist]
        label = f"Iter {r['iteration']}"
        axes[0].plot(epochs, train_loss, lw=1.5, color=colors[idx], label=label)
        axes[1].plot(epochs, val_loss,   lw=1.5, color=colors[idx], label=label, linestyle="--")

    axes[0].set_title("Train KL loss per iteration")
    axes[1].set_title("Val KL loss per iteration")
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Cross-entropy (KL proxy)")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # ── Eval win probability across iterations ─────────────────────────────────
    iters    = [r["iteration"]     for r in log]
    win_probs = [r["eval_win_prob"] for r in log]

    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(iters, win_probs, "o-", color="steelblue", lw=2, ms=7)
    ax.axhline(0.525, color="tomato", lw=1.2, linestyle="--", label="promotion threshold (0.525)")
    ax.axhline(0.5,   color="gray",   lw=0.8, linestyle=":",  label="symmetric baseline (0.5)")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Mean FM win prob vs. previous policy")
    ax.set_title("Policy evaluation: new vs. old across iterations")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("No loss history in log — run training first.")


## Top-5 recommendation shift across iterations

Loads the saved policy snapshots for each iteration and compares recommendations from the same draft state.  At minimum one pick should shift in rank between iteration 0 and iteration 1.

In [ ]:
from bsdraft.selfplay.policy_net import PolicyInference
from bsdraft.mcts.recommend import recommend as mcts_recommend

POLICY_DIR = DATA_DIR / "policy"

# ── Load per-iteration policies ────────────────────────────────────────────────
policies_by_iter: dict[int, object] = {}
for r in log:
    it = r["iteration"]
    p  = POLICY_DIR / f"policy_iter_{it:03d}.pkl"
    if p.exists():
        policies_by_iter[it] = PolicyInference.load(p)

if len(policies_by_iter) < 2:
    print("Need at least 2 iteration snapshots to compare.  Run more training iterations.")
else:
    # Fixed comparison state — first pick, use map[0] from the eval set
    from bsdraft.selfplay.train import select_eval_maps
    eval_maps = select_eval_maps(pairs, n_per_mode=1)
    CMP_MAP, CMP_MODE = eval_maps[2]
    CMP_SKILL = 2.0
    N_CMP_SIMS = 20_000   # keep fast; policy guidance makes 1k sims quite informative

    base_kw = dict(
        my_picks=[], opp_picks=[],
        mode=CMP_MODE, map_name=CMP_MAP, skill_ns=CMP_SKILL,
        is_first_pick=True, n_simulations=N_CMP_SIMS,
        evaluator=evaluator, db=db, n_top=10,
        min_pick_rate=0.01,
    )

    print(f"State: pick 0 | {CMP_MAP} ({CMP_MODE}) | P1 | skill={CMP_SKILL} | {N_CMP_SIMS} sims")
    print()

    iter_results = {}
    for it in sorted(policies_by_iter):
        res = mcts_recommend(**base_kw, policy=policies_by_iter[it],
                             rng=np.random.default_rng(0))
        iter_results[it] = [p["brawler"] for p in res.top_picks[:5]]

    # Side-by-side table
    max_iter = max(iter_results)
    header = f"  {'Rank':<5}" + "".join(f"  {'Iter '+str(it):<18}" for it in sorted(iter_results))
    print(header)
    print("  " + "─" * (5 + 20 * len(iter_results)))
    for rank in range(5):
        row = f"  {rank+1:<5}"
        for it in sorted(iter_results):
            picks = iter_results[it]
            b = picks[rank] if rank < len(picks) else "—"
            row += f"  {b:<18}"
        print(row)

    # Compute how many positions changed between consecutive iterations
    iters_sorted = sorted(iter_results)
    for a, b in zip(iters_sorted, iters_sorted[1:]):
        changed = sum(1 for r in range(5) if iter_results[a][r] != iter_results[b][r])
        print(f"\n  Positions changed iter {a}→{b}: {changed}/5")


---

# Using the Trained Policy in `recommend()`

Load the best promoted policy and use it for a live recommendation.  The policy acts as the PUCT prior at opponent-turn nodes — it guides the MCTS tree toward realistic counter-responses learned from self-play, rather than relying solely on the heuristic counter-rate prior.

In [ ]:
best_policy_path = DATA_DIR / "policy" / "policy_best.pkl"

if not best_policy_path.exists():
    print(f"No trained policy at {best_policy_path}")
    print("Run the training cells above first.")
    trained_policy = None
else:
    trained_policy = PolicyInference.load(best_policy_path)
    n_params = sum(p.numel() for p in trained_policy.net.parameters())
    print(f"Loaded policy from {best_policy_path}  ({n_params:,} parameters)")

    # ── Quick recommendation ───────────────────────────────────────────────────
    if log:
        last_iter = log[-1]
        promoted_iters = [r for r in log if r["promoted"]]
        if promoted_iters:
            print(f"Promoted at iterations: {[r['iteration'] for r in promoted_iters]}")
        else:
            print("No iteration was promoted yet — policy may need more training.")

    rec = mcts_recommend(
        my_picks=["Bo", "Emz", "Otis"], opp_picks=["Crow", "bibi"],
        mode=CMP_MODE, map_name=CMP_MAP, skill_ns=CMP_SKILL,
        is_first_pick=True, n_simulations=30_000,
        evaluator=evaluator, db=db, n_top=10,
        min_pick_rate=0.001,
        policy=trained_policy,
        rng=np.random.default_rng(1),
    )

    print(f"\nRecommendation (trained policy, 30k sims):")
    print(rec.summary())
